# 09d. Train the three matched GAVD JEPAs

This notebook performs fresh, paired-seed pretraining for the standard, paired-unconstrained, and
reflection-equivariant variants. It consumes the frozen contract from `nb_09c`; it does not reuse the
historical one-view checkpoint audited by `nb_09a`.

The default CPU smoke run proves execution only. For a full run, execute `nb_09c` and this notebook with
the same `GAIT_PARITY_MODE`, `GAIT_PARITY_RUN_ID`, profile, seeds, and matching regime.


In [ ]:
from pathlib import Path
import json, math, os, sys

def find_notebook_root(start=None):
    start = Path(start or Path.cwd()).expanduser().resolve()
    relative_path = Path("experiments") / "sjepa" / "gavd6"
    candidates = []
    override = os.getenv("GAIT_PARITY_PROJECT_DIR")
    if override:
        candidates.append(Path(override).expanduser().resolve())
    for base in (start, *start.parents):
        candidates.extend((base, base / relative_path))
    for candidate in dict.fromkeys(candidates):
        if ((candidate / "src" / "gavd6_sjepa" / "research_directions" / "reflection_equivariance" / "jepa_model_architecture.py").is_file()
                and (candidate / "notebooks" / "experiments" / "idea09_reflection_equivariance"
                     / "01_encoder_contract.ipynb").is_file()):
            return candidate
    searched = "\n - ".join(str(path) for path in dict.fromkeys(candidates))
    raise FileNotFoundError(
        "Could not locate experiments/sjepa/gavd6. "
        "Set GAIT_PARITY_PROJECT_DIR to that directory.\n"
        f"Searched:\n - {searched}"
    )

PROJECT_DIR = find_notebook_root()
SOURCE_DIR = PROJECT_DIR / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from gavd6_sjepa.research_directions.reflection_equivariance.jepa_model_architecture import *

MODE = os.getenv("GAIT_PARITY_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAIT_PARITY_MODE must be smoke or real")
PROFILE_NAME = "smoke" if MODE == "smoke" else os.getenv("GAIT_PARITY_PROFILE", "cpu").strip().lower()
if PROFILE_NAME not in PROFILES or (MODE == "real" and PROFILE_NAME == "smoke"):
    raise ValueError("Real runs require GAIT_PARITY_PROFILE=cpu or gpu")
CONFIG = PROFILES[PROFILE_NAME]
MATCHING_REGIME = os.getenv("GAIT_PARITY_MATCHING", "exposure").strip().lower()
if MATCHING_REGIME not in {"exposure", "compute"}:
    raise ValueError("GAIT_PARITY_MATCHING must be exposure or compute")
RUN_ID = os.getenv("GAIT_PARITY_RUN_ID", "smoke")
if MODE == "real" and RUN_ID == "smoke":
    raise ValueError("Set a versioned GAIT_PARITY_RUN_ID for a real run")
SEEDS = [int(value) for value in os.getenv(
    "GAIT_PARITY_SEEDS", "7" if PROFILE_NAME in {"cpu", "smoke"} else "7,19,31"
).split(",")]
OUT_DIR = PROJECT_DIR / "work" / "artifacts" / "gait_parity" / MODE / RUN_ID / MATCHING_REGIME
OUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "smoke":
    RECORDS = synthetic_records(frames=48)
    POSE_DIR = None
else:
    POSE_DIR = resolve_pose_dir(PROJECT_DIR)
    RECORDS = load_gavd_records(POSE_DIR)
WINDOWS, VALID_PATCH, WINDOW_TABLE = build_windows(RECORDS, CONFIG)
MANIFEST = cohort_manifest(RECORDS, WINDOW_TABLE, CONFIG, MODE)

print("scope            :", MANIFEST["scope"])
print("mode/profile     :", MODE, "/", PROFILE_NAME)
print("matching regime  :", MATCHING_REGIME)
print("run ID           :", RUN_ID)
print("device default   :", os.getenv("GAIT_PARITY_DEVICE", "cpu"))
print("records/windows  :", len(RECORDS), "/", len(WINDOWS))
print("source videos    :", MANIFEST["source_video_count"])
print("output           :", OUT_DIR)


## 1. Refuse cohort, configuration, or objective drift


In [ ]:
contract_path = OUT_DIR / "training_contract.json"
if not contract_path.exists():
    raise FileNotFoundError(f"Run nb_09c first: {contract_path}")
CONTRACT = json.loads(contract_path.read_text())
print("Frozen contract accepted:", contract_path)


## 2. Build paired-seed models and freeze update allocations

In exposure mode, batch indices and mask RNG streams are identical across variants for each seed. In
compute mode, variants traverse the same deterministic stream for different prespecified numbers of
updates. Parameter counts are reported, not described as matched: exact weight tying necessarily changes
the number of independent parameters at fixed width.


In [ ]:
allocation_models = {variant: build_model(CONFIG, variant, SEEDS[0]) for variant in VARIANTS}
UPDATES = planned_updates(allocation_models, CONFIG, len(WINDOWS), MATCHING_REGIME)
display(pd.DataFrame({
    variant: {
        "parameters": parameter_count(model),
        "updates": UPDATES[variant],
        "orbit_exposures": UPDATES[variant] * CONFIG.batch_size,
        "compute_proxy_total": UPDATES[variant] * compute_proxy_per_step(model, CONFIG),
    } for variant, model in allocation_models.items()
}).T)


## 3. Train and checkpoint every variant

The EMA teacher receives no gradients. Every checkpoint carries its training configuration,
exposure counts, compute proxy, and measured wall time. CUDA is used only
when `GAIT_PARITY_DEVICE=cuda`; the default remains CPU.


In [ ]:
DEVICE = device_from_environment()
print("training device:", DEVICE)
run_rows = []
for seed in SEEDS:
    for variant in VARIANTS:
        print(f"\n--- seed {seed} | {variant} | {UPDATES[variant]} updates ---")
        model = build_model(CONFIG, variant, seed)
        model, projector, history, wall_seconds = train_variant(
            model, WINDOWS, VALID_PATCH, CONFIG, DEVICE, UPDATES[variant], seed
        )
        stem = f"seed-{seed}_{variant}"
        history_path = OUT_DIR / f"{stem}_history.csv"
        checkpoint_path = OUT_DIR / f"{stem}.pt"
        history.to_csv(history_path, index=False)
        metadata = {
            "variant": variant,
            "seed": seed,
            "train_config": asdict(CONFIG),
            "paired_mask_contract": PAIRED_MASK_CONTRACT,
            "matching_regime": MATCHING_REGIME,
            "optimizer_updates": UPDATES[variant],
            "orbit_exposures": UPDATES[variant] * CONFIG.batch_size,
            "branch_forward_exposures": UPDATES[variant] * CONFIG.batch_size * 8,
            "compute_proxy_per_step": compute_proxy_per_step(model, CONFIG),
            "trainable_parameters": parameter_count(model),
            "wall_clock_seconds": wall_seconds,
            "device": str(DEVICE),
            "amp_enabled": bool(CONFIG.amp and DEVICE.type == "cuda"),
            "scope": MANIFEST["scope"],
        }
        save_checkpoint(checkpoint_path, model, projector, metadata)
        row = {
            **metadata,
            "checkpoint": str(checkpoint_path),
            "history": str(history_path),
            "first_total_loss": float(history.total_loss.iloc[0]),
            "last_total_loss": float(history.total_loss.iloc[-1]),
            "first_jepa_loss": float(history.jepa_loss.iloc[0]),
            "last_jepa_loss": float(history.jepa_loss.iloc[-1]),
        }
        run_rows.append(row)
        print(f"loss {row['first_total_loss']:.4f} -> {row['last_total_loss']:.4f}; {wall_seconds:.1f}s")

training_manifest = {
    "notebook": "nb_09d_gavd_matched_jepa_training",
    "scope": MANIFEST["scope"],
    "contract": str(contract_path),
    "runs": run_rows,
}
write_json(OUT_DIR / "training_manifest.json", training_manifest)
display(pd.DataFrame(run_rows)[["seed", "variant", "optimizer_updates", "orbit_exposures", "trainable_parameters", "wall_clock_seconds", "first_total_loss", "last_total_loss"]])
print("Wrote", OUT_DIR / "training_manifest.json")
